# Part 2.1: Downloading the Hourly Wikipedia Event Data

In [1]:
from pathlib import Path

import boto3
from botocore import UNSIGNED
from botocore.config import Config

BUCKET = "dsan6000-wikipedia"
PREFIX = "hourly_parquet/"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

In [2]:
paginator = s3.get_paginator("list_objects_v2")
keys = [
    obj["Key"]
    for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX)
    for obj in page.get("Contents", [])
    if obj["Key"].endswith(".parquet")
]
keys.sort()
print(len(keys), "files")
keys[:5]

24 files


['hourly_parquet/20260901_040000.parquet',
 'hourly_parquet/20260901_050000.parquet',
 'hourly_parquet/20260901_060000.parquet',
 'hourly_parquet/20260901_070000.parquet',
 'hourly_parquet/20260901_080000.parquet']

In [3]:
for key in keys:
    s3.download_file(BUCKET, key, str(DATA_DIR / Path(key).name))

local_files = sorted(DATA_DIR.glob("*.parquet"))
print(len(local_files), "files downloaded to", DATA_DIR.resolve())
print(f"{sum(f.stat().st_size for f in local_files) / 1024**2:.1f} MB")

24 files downloaded to /home/ubuntu/dsan6000-hw02-jupyter-on-ec2/data
30.5 MB
